# Prediction Request Testing — Cloud Endpoint

Notebook ini digunakan untuk menguji dan melakukan prediction request ke sistem machine learning yang telah di-deploy di **Railway**.

**Endpoint:** `https://<RAILWAY_APP_URL>/v1/models/churn-model:predict`

In [17]:
import requests
import json
import numpy as np
import tensorflow as tf

SERVING_URL = "https://kendrickfff-submission2-production.up.railway.app/v1/models/churn-model"
PREDICT_URL = f"{SERVING_URL}:predict"
print(f"Model status endpoint: {SERVING_URL}")
print(f"Prediction endpoint: {PREDICT_URL}")

Model status endpoint: https://kendrickfff-submission2-production.up.railway.app/v1/models/churn-model
Prediction endpoint: https://kendrickfff-submission2-production.up.railway.app/v1/models/churn-model:predict


## 1. Check Model Status

Pastikan model sudah aktif dan siap menerima request.

In [18]:
# Check model status
response = requests.get(SERVING_URL, timeout=10)
print(f"Status Code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

Status Code: 200
Response: {
  "model_version_status": [
    {
      "version": "1773862247",
      "state": "AVAILABLE",
      "status": {
        "error_code": "OK",
        "error_message": ""
      }
    }
  ]
}


## 2. Helper Functions

Fungsi bantuan untuk membuat serialized tf.Example dan mengirim prediction request.

In [19]:
def create_tf_example(customer_data):
    """Create a serialized tf.Example from customer data dictionary."""
    feature = {}

    # Integer features — tenure and SeniorCitizen are int64
    for key in ["tenure", "SeniorCitizen"]:
        feature[key] = tf.train.Feature(
            int64_list=tf.train.Int64List(value=[int(customer_data[key])])
        )

    # Float features
    for key in ["MonthlyCharges", "TotalCharges"]:
        feature[key] = tf.train.Feature(
            float_list=tf.train.FloatList(value=[float(customer_data[key])])
        )

    # Categorical features
    for key in ["customerID", "gender", "Partner", "Dependents", "PhoneService",
                "InternetService", "Contract", "PaperlessBilling", "PaymentMethod"]:
        feature[key] = tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[customer_data[key].encode()])
        )

    example = tf.train.Example(
        features=tf.train.Features(feature=feature)
    )

    import base64
    return base64.b64encode(example.SerializeToString()).decode("utf-8")


## 3. Test Predictions

Mengirim beberapa sample prediction request dengan skenario berbeda.

In [21]:
# Test Case 1: Pelanggan dengan risiko churn TINGGI
# - Kontrak month-to-month, tenure rendah, charges tinggi
high_risk_customer = {
    "customerID": "TEST-001",
    "gender": "Male",
    "SeniorCitizen": 1,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 2,
    "PhoneService": "Yes",
    "InternetService": "Fiber optic",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 95.50,
    "TotalCharges": 191.00,
}

result = predict(high_risk_customer)
print("🔴 HIGH RISK Customer:")
print(f"   Prediction: {result}")
if "predictions" in result:
    prob = result["predictions"][0][0]
    print(f"   Churn Probability: {prob:.2%}")
    print(f"   Classification: {'CHURN' if prob > 0.5 else 'NOT CHURN'}")

🔴 HIGH RISK Customer:
   Prediction: {'predictions': [[0.879349113]]}
   Churn Probability: 87.93%
   Classification: CHURN


In [22]:
# Test Case 2: Pelanggan dengan risiko churn RENDAH
# - Kontrak 2 tahun, tenure tinggi, charges moderate
low_risk_customer = {
    "customerID": "TEST-002",
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "Yes",
    "tenure": 60,
    "PhoneService": "Yes",
    "InternetService": "DSL",
    "Contract": "Two year",
    "PaperlessBilling": "No",
    "PaymentMethod": "Bank transfer (automatic)",
    "MonthlyCharges": 55.25,
    "TotalCharges": 3315.00,
}

result = predict(low_risk_customer)
print("🟢 LOW RISK Customer:")
print(f"   Prediction: {result}")
if "predictions" in result:
    prob = result["predictions"][0][0]
    print(f"   Churn Probability: {prob:.2%}")
    print(f"   Classification: {'CHURN' if prob > 0.5 else 'NOT CHURN'}")

🟢 LOW RISK Customer:
   Prediction: {'predictions': [[0.0120454477]]}
   Churn Probability: 1.20%
   Classification: NOT CHURN


In [23]:
# Test Case 3: Pelanggan dengan risiko MEDIUM
# - Kontrak 1 tahun, tenure moderate
medium_risk_customer = {
    "customerID": "TEST-003",
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 24,
    "PhoneService": "Yes",
    "InternetService": "Fiber optic",
    "Contract": "One year",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Credit card (automatic)",
    "MonthlyCharges": 78.90,
    "TotalCharges": 1893.60,
}

result = predict(medium_risk_customer)
print("🟡 MEDIUM RISK Customer:")
print(f"   Prediction: {result}")
if "predictions" in result:
    prob = result["predictions"][0][0]
    print(f"   Churn Probability: {prob:.2%}")
    print(f"   Classification: {'CHURN' if prob > 0.5 else 'NOT CHURN'}")

🟡 MEDIUM RISK Customer:
   Prediction: {'predictions': [[0.114277922]]}
   Churn Probability: 11.43%
   Classification: NOT CHURN


## 4. Batch Prediction Test

Menguji batch prediction dengan beberapa pelanggan sekaligus.

In [25]:
# Batch prediction - multiple customers
test_customers = [
    {"name": "Customer A (New, Month-to-month)", "data": high_risk_customer},
    {"name": "Customer B (Loyal, Two year)", "data": low_risk_customer},
    {"name": "Customer C (Mid-term, One year)", "data": medium_risk_customer},
]

print("Batch Prediction Results:")
print("-" * 60)
for customer in test_customers:
    result = predict(customer["data"])
    if "predictions" in result:
        prob = result["predictions"][0][0]
        status = "🔴 CHURN" if prob > 0.5 else "🟢 RETAIN"
        print(f"{customer['name']:40s} | {prob:.2%} | {status}")
    else:
        print(f"{customer['name']:40s} | ERROR: {result}")
print("-" * 60)

Batch Prediction Results:
------------------------------------------------------------
Customer A (New, Month-to-month)         | 87.93% | 🔴 CHURN
Customer B (Loyal, Two year)             | 1.20% | 🟢 RETAIN
Customer C (Mid-term, One year)          | 11.43% | 🟢 RETAIN
------------------------------------------------------------


## 5. Monitoring Endpoint Check

Verifikasi bahwa Prometheus metrics endpoint aktif.

In [27]:
METRICS_URL = f"{SERVING_URL.rsplit('/v1', 1)[0]}/monitoring/prometheus/metrics"
print(f"Metrics endpoint: {METRICS_URL}")

try:
    response = requests.get(METRICS_URL, timeout=10)
    print(f"Status Code: {response.status_code}")
    if response.status_code == 200:
        metrics_lines = response.text.strip().split("\n")
        print(f"\nTotal metrics lines: {len(metrics_lines)}")
        print("\nSample metrics:")
        for line in metrics_lines[:15]:
            if not line.startswith("#"):
                print(f"  {line}")
    else:
        print(f"Note: Monitoring endpoint not available on Railway (status {response.status_code})")
        print(f"This is expected - Prometheus monitoring works locally with docker-compose setup")
        print(f"See monitoring/ folder for local Prometheus + Grafana configuration")
except Exception as e:
    print(f"Error: {e}")

Metrics endpoint: https://kendrickfff-submission2-production.up.railway.app/monitoring/prometheus/metrics
Status Code: 404
Note: Monitoring endpoint not available on Railway (status 404)
This is expected - Prometheus monitoring works locally with docker-compose setup
See monitoring/ folder for local Prometheus + Grafana configuration


## ✅ Kesimpulan

Model churn prediction telah berhasil di-deploy ke cloud (Railway) dan dapat menerima prediction request melalui REST API. Prometheus metrics endpoint juga aktif untuk monitoring.